## Windows setup notes

If you are running this notebook on Windows, the main thing that may need changing is the FFmpeg and FFprobe path.

The notebook may currently use Mac-specific paths like:

`ffmpeg_path="/opt/homebrew/opt/ffmpeg-full/bin/ffmpeg"`

`ffprobe_path="/opt/homebrew/opt/ffmpeg-full/bin/ffprobe"`

If FFmpeg is installed and available in Command Prompt or PowerShell, change the paths to:

`ffmpeg_path="ffmpeg"`

`ffprobe_path="ffprobe"`

If FFmpeg is not on PATH, use the full path to the .exe files, for example:

`ffmpeg_path=r"C:\ffmpeg\bin\ffmpeg.exe"`

`ffprobe_path=r"C:\ffmpeg\bin\ffprobe.exe"`

To confirm FFmpeg works on Windows, open Command Prompt and run:

`ffmpeg -version`

`ffprobe -version`

If those commands work, you can usually just use:

`ffmpeg_path="ffmpeg"`

`ffprobe_path="ffprobe"`

In [ ]:
from pathlib import Path
import subprocess
import json

In [ ]:
def get_video_duration(video_path, ffprobe_path="/opt/homebrew/opt/ffmpeg-full/bin/ffprobe"):
    video_file = Path(video_path)

    if not video_file.exists():
        raise FileNotFoundError(f"video file not found: {video_file}")

    cmd = [
        ffprobe_path,
        "-v", "error",
        "-show_entries", "format=duration",
        "-of", "json",
        str(video_file)
    ]

    result = subprocess.run(cmd, capture_output=True, text=True)

    if result.returncode != 0:
        print(result.stderr)
        raise RuntimeError("ffprobe failed while reading video duration")

    data = json.loads(result.stdout)
    return float(data["format"]["duration"])

In [ ]:
def make_video_fit_2_minutes(
    video_path,
    output_path=None,
    target_seconds=120,
    ffmpeg_path="/opt/homebrew/opt/ffmpeg-full/bin/ffmpeg",
    ffprobe_path="/opt/homebrew/opt/ffmpeg-full/bin/ffprobe"
):
    video_file = Path(video_path)

    if not video_file.exists():
        raise FileNotFoundError(f"video file not found: {video_file}")

    if output_path is None:
        output_path = str(video_file.with_name(video_file.stem + "_2min_noaudio.mp4"))

    duration = get_video_duration(video_file, ffprobe_path=ffprobe_path)

    # setpts multiplier:
    # new_duration = old_duration * multiplier
    # so multiplier = target / original
    speed_factor = target_seconds / duration

    print(f"original duration: {duration:.2f} seconds")
    print(f"target duration:   {target_seconds:.2f} seconds")
    print(f"setpts factor:     {speed_factor:.6f}")

    cmd = [
        ffmpeg_path,
        "-y",
        "-i", str(video_file),
        "-an",                         # remove audio
        "-filter:v", f"setpts={speed_factor}*PTS",
        "-movflags", "+faststart",
        output_path
    ]

    print("\nrunning command:")
    print(" ".join(cmd))

    result = subprocess.run(cmd, capture_output=True, text=True)

    print("\n--- ffmpeg stderr ---")
    print(result.stderr)

    if result.returncode != 0:
        raise RuntimeError(f"ffmpeg failed with exit code {result.returncode}")

    print(f"\ndone. output saved to: {output_path}")
    return output_path

In [ ]:
output_video = make_video_fit_2_minutes(
    video_path="my_video.mp4"
)

output_video